# Create composite images
Build large composite images combining GFW and OBIS layers on the base map.

In [8]:
%pip install duckdb datashader colorcet 

Note: you may need to restart the kernel to use updated packages.


In [3]:
import duckdb
import datashader as ds
import datashader.transfer_functions as tf
import colorcet as cc
from PIL import Image, ImageDraw, ImageFont
import os

Image.MAX_IMAGE_PIXELS = None

In [5]:
YEAR_START = "2012-01"
YEAR_END = "2026-12"
SPECIES = ["*"]
OUTPUT_DIR = "/mnt/shared_data/finflow/images/"
W, H = 6000, 3000

In [3]:
    cvs = ds.Canvas(plot_width=W, plot_height=H, x_range=(-180, 180), y_range=(-90, 90))
    
    paths_gfw = "/mnt/shared_data/finflow/gfw_raw/*/*.parquet"
    
    query_gfw = f"""
    SELECT lon, lat, hours 
    FROM read_parquet('{paths_gfw}', filename=true)
    WHERE regexp_extract(filename, '(\d{{4}}-\d{{2}})') BETWEEN '{YEAR_START}' AND '{YEAR_END}'
    """
    
    df_gfw = duckdb.query(query_gfw).to_df()
    
    agg_gfw = cvs.points(df_gfw, 'lon', 'lat', ds.sum('hours'))
    img_gfw = tf.shade(agg_gfw, cmap=cc.fire, how='log').to_pil().convert("RGBA")
    
    for spec in SPECIES:
        path_obis = f"/mnt/shared_data/finflow/obis_raw/{spec}/*/*.parquet"
    
        query_obis = f"""
        SELECT decimalLongitude as lon, decimalLatitude as lat 
        FROM read_parquet('{path_obis}') 
        WHERE lon IS NOT NULL AND lat IS NOT NULL
        """
        
        df_obis = duckdb.query(query_obis).to_df()
        
        combined = Image.open("/mnt/shared_data/finflow/images/base_map.png").resize((W, H)).convert("RGBA")
        combined.alpha_composite(img_gfw)
        
        if not df_obis.empty:
            agg_obis = cvs.points(df_obis, 'lon', 'lat', ds.count())
            img_obis = tf.shade(agg_obis, cmap=["#90ee90", "#00ff00"], how='log').to_pil().convert("RGBA")
            combined.alpha_composite(img_obis)
        
        file_spec = spec.replace("*", "all").replace(" ", "_")
        save_path = os.path.join(OUTPUT_DIR, f"{file_spec}_{YEAR_START}_to_{YEAR_END}.png")
        combined.save(save_path)
        print(f"Successfully saved: {save_path}")

NameError: name 'ds' is not defined

In [16]:
import duckdb
import datashader as ds
import datashader.transfer_functions as tf
import colorcet as cc
from PIL import Image, ImageDraw, ImageFont
import os
import gc

Image.MAX_IMAGE_PIXELS = None

# --- Konfiguration ---
YEAR_START = "2012-01"
YEAR_END = "2026-12"
BASE_PATH_OBIS = "/mnt/shared_data/finflow/obis_raw/"
OUTPUT_DIR = "/mnt/shared_data/finflow/images/"
W, H = 5000, 2500
LEGEND_WIDTH = 1800 

# Pfad zu deiner hochgeladenen Schriftart
CUSTOM_FONT_PATH = os.path.join(OUTPUT_DIR, "font.ttf")

# --- 1. Mapping & Farben ---
species_info = {
    "Scombridae":       {"en": "Mackerels & Tunas",    "color": "#00dbff"},
    "Cephalopoda":      {"en": "Squids & Octopuses",   "color": "#ff00ff"},
    "Cheloniidae":      {"en": "Sea Turtles",          "color": "#00ff00"},
    "Pleuronectiformes": {"en": "Flatfish",            "color": "#0080ff"},
    "Otariidae":        {"en": "Eared Seals",          "color": "#adff2f"},
    "Elasmobranchii":   {"en": "Sharks & Rays",        "color": "#00ffff"},
    "Xiphiidae":        {"en": "Swordfish",            "color": "#ff007f"},
    "Gadidae":          {"en": "Cods & Haddocks",      "color": "#b0e0e6"},
    "Sirenia":          {"en": "Manatees & Dugongs",   "color": "#7fffd4"},
    "Cetacea":          {"en": "Whales & Dolphins",    "color": "#ffffff"},
    "Holocephali":      {"en": "Chimaeras",            "color": "#da70d6"},
    "Sphenisciformes":  {"en": "Penguins",             "color": "#1e90ff"},
    "Dermochelyidae":   {"en": "Leatherback Turtles",  "color": "#32cd32"},
    "Clupeidae":        {"en": "Herrings & Sardines",  "color": "#8a2be2"},
    "Istiophoridae":    {"en": "Billfish (Marlins)",   "color": "#00fa9a"},
    "Phocidae":         {"en": "Earless Seals",        "color": "#ee82ee"},
    "Odobenidae":       {"en": "Walruses",             "color": "#40e0d0"}
}

# --- 2. Sortierung (Häufigste nach unten) ---
print("Sortiere Layer nach Datenmenge...")
species_counts = []
for spec in species_info.keys():
    path = f"{BASE_PATH_OBIS}{spec}/*/*.parquet"
    if os.path.exists(os.path.dirname(path.split('*')[0])):
        count = duckdb.query(f"SELECT count(*) FROM read_parquet('{path}')").fetchone()[0]
        species_counts.append((spec, count))

species_counts.sort(key=lambda x: x[1], reverse=True)
SORTED_SPECIES = [x[0] for x in species_counts]

# --- 3. GFW & Basiskarte ---
cvs = ds.Canvas(plot_width=W, plot_height=H, x_range=(-180, 180), y_range=(-90, 90))
paths_gfw = "/mnt/shared_data/finflow/gfw_raw/*/*.parquet"
query_gfw = f"SELECT lon, lat, hours FROM read_parquet('{paths_gfw}', filename=true) WHERE regexp_extract(filename, '(\d{{4}}-\d{{2}})') BETWEEN '{YEAR_START}' AND '{YEAR_END}'"

df_gfw = duckdb.query(query_gfw).to_df()
agg_gfw = cvs.points(df_gfw, 'lon', 'lat', ds.sum('hours'))
img_gfw = tf.shade(agg_gfw, cmap=cc.fire, how='log').to_pil().convert("RGBA")
combined = Image.open("/mnt/shared_data/finflow/images/base_map.png").resize((W, H)).convert("RGBA")
combined.alpha_composite(img_gfw)
del df_gfw, agg_gfw, img_gfw
gc.collect()

# --- 4. OBIS Layer hinzufügen ---
for spec in SORTED_SPECIES:
    path_obis = f"{BASE_PATH_OBIS}{spec}/*/*.parquet"
    df_obis = duckdb.query(f"SELECT decimalLongitude as lon, decimalLatitude as lat FROM read_parquet('{path_obis}') WHERE lon IS NOT NULL AND lat IS NOT NULL").to_df()
    if not df_obis.empty:
        agg_obis = cvs.points(df_obis, 'lon', 'lat', ds.count())
        img_obis = tf.shade(agg_obis, cmap=[species_info[spec]['color']], how='linear').to_pil().convert("RGBA")
        combined.alpha_composite(img_obis)
        del agg_obis, img_obis
    del df_obis
    gc.collect()

# --- 5. Legende mit Custom Font & angepassten Abständen ---
final_img = Image.new("RGBA", (W + LEGEND_WIDTH, H), (10, 10, 15, 255))
final_img.paste(combined, (0, 0))
del combined

draw = ImageDraw.Draw(final_img)

if os.path.exists(CUSTOM_FONT_PATH):
    print(f"Lade Custom Font: {CUSTOM_FONT_PATH}")
    font_title = ImageFont.truetype(CUSTOM_FONT_PATH, 80) # Titel-Größe
    font_en    = ImageFont.truetype(CUSTOM_FONT_PATH, 60)  # Englisch-Größe
    font_sci   = ImageFont.truetype(CUSTOM_FONT_PATH, 40)  # Wissenschaftlich-Größe
else:
    print("!!! FEHLER: font.ttf wurde im Output-Pfad nicht gefunden !!!")
    font_title = font_en = font_sci = ImageFont.load_default()

# Startposition Legende
x_offset = W + 100
y_offset = 120

# Titel zeichnen
draw.text((x_offset, y_offset), "Cargo Traffic (Red) and Marine Species", fill="white", font=font_title)
y_offset += 180  # Kleinerer Abstand nach dem Titel (vorher 400)

for spec in SORTED_SPECIES:
    info = species_info[spec]
    box_sz = 100  # Box etwas kleiner gemacht (vorher 160), damit sie zur 60pt Schrift passt
    
    # Farbbox zeichnen
    draw.rectangle([x_offset, y_offset, x_offset + box_sz, y_offset + box_sz], 
                   fill=info['color'], outline="white", width=4)
    
    # Text-Positionierung angepasst an kleinere Box und Font
    # Englischer Name: leicht nach rechts versetzt, vertikal bündig mit Box-Oberkante
    draw.text((x_offset + box_sz + 40, y_offset - 5), info['en'], fill="white", font=font_en)
    
    # Wissenschaftlicher Name: direkt unter den englischen Namen
    draw.text((x_offset + box_sz + 40, y_offset + 65), f"({spec})", fill="#cccccc", font=font_sci)
    
    # Abstand zum nächsten Eintrag (100px Box + 60px Lücke)
    y_offset += 160 
    
    if y_offset > H - 150: break

# --- 6. Save ---
save_path = os.path.join(OUTPUT_DIR, f"final_map.png")
final_img.save(save_path)
print(f"Erfolgreich gespeichert unter: {save_path}")

Sortiere Layer nach Datenmenge...
Lade Custom Font: /mnt/shared_data/finflow/images/font.ttf
Erfolgreich gespeichert unter: /mnt/shared_data/finflow/images/final_map.png
